In [ ]:
# Core scientific computing
import numpy as np
import pandas as pd
from scipy import linalg
from scipy.signal import find_peaks
from scipy.integrate import simpson
from scipy.ndimage import gaussian_filter1d
from scipy.fft import fft, fftfreq, fftshift

# Visualization
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns

# System utilities
import os
import sys
from pathlib import Path
from importlib import reload
import warnings
warnings.filterwarnings('ignore')

# Make the project root importable
project_root = Path.cwd().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import our NMR processing functions
from nuclear_magnetic_resonance_spectrospy import nmr_function as nmr

# Set visualization defaults
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_palette("husl")

print("✓ All imports successful!")
print(f"✓ NMR functions loaded from: {nmr.__file__}")
print(f"✓ Working directory: {os.getcwd()}")
print(f"✓ NumPy version: {np.__version__}")

# Section 0: Setup & Imports

Let's import all the libraries we'll need for quantum mechanics calculations and visualizations.

# Quantum Mechanics in Nuclear Magnetic Resonance Spectroscopy

## A Visual Learning Journey from NMR Data to Quantum Entanglement

This notebook explores the deep quantum mechanical foundations of NMR spectroscopy. We'll start by analyzing real hydrogen NMR data, extract quantum observables from the spectrum, and then systematically build up the quantum mechanical framework that explains what we observe.

### What You'll Learn

1. **Data Pre-Stage**: Load and process hydrogen NMR FID data to extract quantum-relevant information
2. **Quantum States**: Understanding spin-1/2 systems, superposition, and state spaces
3. **Tensor Products**: Building multi-spin systems from single spins
4. **Density Matrices**: Describing pure and mixed quantum states
5. **Entanglement**: From separable to maximally entangled spin pairs
6. **Bell's Theorem**: Quantum correlations that violate classical physics
7. **Heisenberg Uncertainty**: Fundamental limits on simultaneous measurements
8. **NMR Connection**: How all these quantum concepts manifest in your spectrum

### Learning Approach

- **Visual First**: Heavy emphasis on plots, heatmaps, 3D visualizations
- **Data-Driven**: Every concept applied to real NMR measurements
- **Computational**: Calculate density matrices, entanglement measures, Bell violations
- **Progressive**: Build from simple single spins to complex entangled systems

Let's begin! 🚀

# Section 1: Data Pre-Stage - Loading & Processing NMR Data

## 1.1 The Free Induction Decay (FID)

Before we dive into quantum mechanics, we need to extract quantum observables from real NMR data. Nuclear spins in a magnetic field precess at their Larmor frequency, and this precession is what we measure in NMR.

### What is the FID?

The **Free Induction Decay (FID)** is the time-domain signal recorded after a radiofrequency pulse excites nuclear spins. It contains:
- **Frequencies**: Encode chemical shifts (different nuclear environments)
- **Amplitudes**: Reflect population differences between spin states
- **Decay rates**: Related to relaxation (decoherence) times
- **Phases**: Real and imaginary components encode quantum coherences

### Connection to Quantum Mechanics

Each peak in the NMR spectrum corresponds to a specific transition between quantum energy levels. The spacing between peaks (J-coupling) directly measures the quantum mechanical interaction between spins.

Let's load the hydrogen NMR data and see what quantum information it contains!

In [ ]:
# Load the hydrogen FID data
url = r"https://raw.githubusercontent.com/Quintinlf/NMR-Project/main/spring_semester_2025/13_03_11_indst_1H%20fid.asc"

print("Loading FID data from GitHub...")
df, name = nmr.load_fid_and_preview(url)
data = df.to_numpy() if hasattr(df, "to_numpy") else np.asarray(df)

print(f"\n✓ Loaded: {name}")
print(f"✓ Data shape: {data.shape}")
print(f"✓ Time points: {len(data)}")
print(f"✓ Acquisition time: {data[-1, 0]:.4f} seconds")

In [ ]:
# Visualize the time-domain FID signal
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Real component (observable signal)
ax1.plot(data[:, 0], data[:, 1], color='steelblue', linewidth=0.8)
ax1.set_xlabel('Time (s)', fontsize=12)
ax1.set_ylabel('Signal Amplitude (Real)', fontsize=12)
ax1.set_title('FID: Real Component (Observable)', fontsize=13, fontweight='bold')
ax1.grid(alpha=0.3)

# Imaginary component (phase information)
ax2.plot(data[:, 0], data[:, 2], color='coral', linewidth=0.8)
ax2.set_xlabel('Time (s)', fontsize=12)
ax2.set_ylabel('Signal Amplitude (Imaginary)', fontsize=12)
ax2.set_title('FID: Imaginary Component (Phase)', fontsize=13, fontweight='bold')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("📊 The FID shows the free precession of nuclear spins after excitation.")
print("📊 The decay reflects quantum decoherence (T2 relaxation).")

## 1.2 Fourier Transform: Time → Frequency Domain

The Fourier transform converts the time-domain FID into a frequency-domain spectrum. This reveals the **energy level transitions** for each chemically distinct nucleus.

### Quantum Interpretation

$$\text{FID}(t) = \sum_i A_i e^{i\omega_i t} e^{-t/T_2} \quad \xrightarrow{\mathcal{F}} \quad \text{Spectrum}(\omega)$$

Where:
- $\omega_i = \gamma B_0 (1 - \sigma_i)$ is the Larmor frequency (chemical shift)
- $A_i$ is proportional to the population difference: $A_i \propto (n_\alpha - n_\beta)$
- $T_2$ is the decoherence time (transverse relaxation)

Each frequency peak corresponds to a **quantum transition** between spin states $|\alpha\rangle$ and $|\beta\rangle$.

In [ ]:
# Compute FFT spectrum
print("Computing Fourier Transform...")
fft_res = nmr.compute_fft_spectrum(data)
frequencies = fft_res["frequencies"]
magnitude = fft_res["magnitude"]

# Spectrometer frequency (determines field strength B₀)
spectrometer_freq = 399.78219838  # MHz
print(f"✓ Spectrometer frequency: {spectrometer_freq} MHz")
print(f"✓ Magnetic field B₀ ≈ {spectrometer_freq / 42.577:.2f} Tesla (for ¹H)")

# Convert to ppm (parts per million) - the standard NMR unit
ppm_axis = frequencies / spectrometer_freq

print(f"✓ FFT complete: {len(frequencies)} frequency points")
print(f"✓ Spectral width: {ppm_axis.max() - ppm_axis.min():.2f} ppm")

In [ ]:
# Plot the full NMR spectrum
plt.figure(figsize=(14, 6))
plt.plot(ppm_axis, magnitude, linewidth=0.8, color='darkblue')
plt.gca().invert_xaxis()  # NMR convention: high ppm (deshielded) on left
plt.xlabel('Chemical Shift δ (ppm)', fontsize=13, fontweight='bold')
plt.ylabel('Intensity (arbitrary units)', fontsize=13, fontweight='bold')
plt.title('¹H NMR Spectrum - Frequency Domain', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3, linestyle='--')
plt.xlim(12, 0)  # Typical ¹H range
plt.tight_layout()
plt.show()

print("📊 Each peak represents a quantum transition for protons in different chemical environments.")
print("📊 Peak positions (ppm) encode electron shielding → chemical structure.")
print("📊 Peak intensities encode population differences → number of equivalent spins.")

## 1.3 Extracting Quantum Components: Peak Detection & J-Coupling

Now we'll extract the quantum observables from the spectrum:

1. **Peak positions (δ)**: Chemical shifts → Different nuclear environments
2. **Peak integrals**: Proportional to number of equivalent spins
3. **J-coupling constants**: Measure quantum interaction strength between spins

### Quantum Significance of J-Coupling

The **J-coupling constant** measures the indirect spin-spin interaction mediated through bonding electrons:

$$H_{J} = 2\pi J \, \mathbf{I}_1 \cdot \mathbf{I}_2 = 2\pi J (I_{1x}I_{2x} + I_{1y}I_{2y} + I_{1z}I_{2z})$$

This interaction **creates entanglement** between coupled spins! The splitting pattern (doublet, triplet, quartet) directly reveals the number of coupled neighbors and the strength of quantum correlation.

In [ ]:
# Detect peaks in the spectrum
print("Detecting peaks and functional groups...")

# Use the nmr_function module to detect peaks and identify groups
ppm_shifts = nmr.PPM_SHIFT_DEFAULTS
result = nmr.plot_full_and_zoom_with_peaks(
    frequencies, magnitude,
    title=f"{name} - Quantum Peak Analysis",
    ppm_shifts=ppm_shifts,
    identify_functional_groups=nmr.identify_functional_groups,
    show=True
)

# Extract results
peaks = result.get("peaks", np.array([]))
ppm_detected = result.get("ppm_axis", ppm_axis)[peaks]
intensities = result.get("intensity", magnitude)[peaks]
identified_groups = result.get("identified_groups", [])

print(f"\n✓ Detected {len(peaks)} significant peaks")
print(f"✓ Identified {len(identified_groups)} functional groups")

if identified_groups:
    print("\n🔬 Quantum Systems Identified:")
    for ppm, group in identified_groups:
        print(f"   • δ = {ppm:.2f} ppm: {group}")

In [ ]:
# Helper functions for J-coupling analysis
def detect_multiplet_structure(ppm_axis, intensity, center_ppm, window=0.05):
    """Detect sub-peaks within a multiplet to measure J-coupling"""
    mask = (ppm_axis > center_ppm - window) & (ppm_axis < center_ppm + window)
    region_ppm = ppm_axis[mask]
    region_intensity = intensity[mask]
    
    # Smooth to reduce noise
    region_intensity = gaussian_filter1d(region_intensity, sigma=2)
    
    # Find sub-peaks
    sub_peaks, _ = find_peaks(region_intensity, 
                               height=0.15 * region_intensity.max(),
                               prominence=0.1 * region_intensity.max())
    
    sub_ppms = np.sort(region_ppm[sub_peaks])
    
    # Calculate J-couplings between adjacent sub-peaks
    j_couplings_hz = []
    for i in range(1, len(sub_ppms)):
        delta_ppm = abs(sub_ppms[i] - sub_ppms[i-1])
        j_hz = delta_ppm * spectrometer_freq  # Convert ppm to Hz
        j_couplings_hz.append(j_hz)
    
    return sub_ppms, j_couplings_hz

# Store quantum data for later use
quantum_data = {
    'peaks_ppm': ppm_detected,
    'intensities': intensities,
    'groups': identified_groups,
    'j_couplings': {},  # Will populate with multiplet analysis
    'spectrometer_freq_MHz': spectrometer_freq,
    'B0_tesla': spectrometer_freq / 42.577  # For ¹H
}

print("✓ Quantum data structure prepared")
print(f"✓ Magnetic field strength: B₀ = {quantum_data['B0_tesla']:.2f} T")

In [ ]:
# Analyze multiplet structure for each identified group
print("\n🔍 Analyzing J-Coupling (Quantum Spin-Spin Interaction):\n")

for ppm_val, group_name in identified_groups[:3]:  # Analyze first 3 groups
    sub_ppms, j_values = detect_multiplet_structure(ppm_axis, magnitude, ppm_val, window=0.08)
    
    if len(j_values) > 0:
        quantum_data['j_couplings'][group_name] = {
            'center_ppm': ppm_val,
            'sub_peaks': sub_ppms,
            'J_couplings_Hz': j_values,
            'n_peaks': len(sub_ppms),
            'J_avg_Hz': np.mean(j_values)
        }
        
        print(f"Group: {group_name} (δ = {ppm_val:.2f} ppm)")
        print(f"  Multiplet: {len(sub_ppms)} peaks detected")
        print(f"  J-coupling: {np.mean(j_values):.2f} ± {np.std(j_values):.2f} Hz")
        print(f"  → Quantum interaction strength: 2πJ = {2*np.pi*np.mean(j_values):.2f} rad/s")
        print(f"  → This creates ENTANGLEMENT between coupled spins!\n")

print(f"✓ Total J-coupled systems detected: {len(quantum_data['j_couplings'])}")

# Section 2: Quantum Foundations - Spin-1/2 Systems

## 2.1 Spin Operators and the Pauli Matrices

In NMR, we're working with **spin-1/2** nuclei (like ¹H, ¹³C, ³¹P). These are the simplest quantum systems, described by **2-dimensional Hilbert space**.

### The Pauli Spin Operators

The spin angular momentum operators are represented by the **Pauli matrices**:

$$\hat{I}_x = \frac{1}{2}\begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}, \quad
\hat{I}_y = \frac{1}{2}\begin{pmatrix} 0 & -i \\ i & 0 \end{pmatrix}, \quad
\hat{I}_z = \frac{1}{2}\begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}$$

These operators satisfy the **angular momentum commutation relations**:

$$[\hat{I}_x, \hat{I}_y] = i\hat{I}_z, \quad [\hat{I}_y, \hat{I}_z] = i\hat{I}_x, \quad [\hat{I}_z, \hat{I}_x] = i\hat{I}_y$$

### Physical Interpretation

- $\hat{I}_z$: Spin projection along the magnetic field (B₀ direction)
- $\hat{I}_x, \hat{I}_y$: Transverse spin components (what we detect in NMR!)
- Eigenvalues of $\hat{I}_z$: $\pm\frac{1}{2}$ (spin "up" α or "down" β)

### Connection to NMR

The Larmor frequency is $\omega_0 = \gamma B_0$, where $\gamma$ is the gyromagnetic ratio. The energy difference between spin states is:

$$\Delta E = \hbar \omega_0 = \gamma \hbar B_0$$

This is exactly what we measure in our spectrum!

In [ ]:
# Define Pauli spin-1/2 operators
I = np.eye(2, dtype=complex)  # Identity
Ix = 0.5 * np.array([[0, 1], [1, 0]], dtype=complex)
Iy = 0.5 * np.array([[0, -1j], [1j, 0]], dtype=complex)
Iz = 0.5 * np.array([[1, 0], [0, -1]], dtype=complex)

# Ladder operators (useful for transitions)
I_plus = Ix + 1j*Iy  # Raises spin: |↓⟩ → |↑⟩
I_minus = Ix - 1j*Iy  # Lowers spin: |↑⟩ → |↓⟩

print("Spin-1/2 Operators Defined:\n")
print("Ix (transverse):")
print(Ix)
print("\nIy (transverse):")
print(Iy)
print("\nIz (along B₀):")
print(Iz)

# Verify commutation relations
comm_xy = Ix @ Iy - Iy @ Ix
comm_yz = Iy @ Iz - Iz @ Iy
comm_zx = Iz @ Ix - Ix @ Iz

print("\n✓ Commutation Relations:")
print(f"[Ix, Iy] = i*Iz? {np.allclose(comm_xy, 1j*Iz)}")
print(f"[Iy, Iz] = i*Ix? {np.allclose(comm_yz, 1j*Ix)}")
print(f"[Iz, Ix] = i*Iy? {np.allclose(comm_zx, 1j*Iy)}")

## 2.2 Basis States and Quantum Superposition

### The Computational Basis

For spin-1/2, we have two basis states (eigenstates of $\hat{I}_z$):

$$|\alpha\rangle = |\uparrow\rangle = \begin{pmatrix} 1 \\ 0 \end{pmatrix}, \quad |\beta\rangle = |\downarrow\rangle = \begin{pmatrix} 0 \\ 1 \end{pmatrix}$$

These correspond to:
- $|\alpha\rangle$: Spin "up" ($m = +\frac{1}{2}$), **lower energy** in field
- $|\beta\rangle$: Spin "down" ($m = -\frac{1}{2}$), **higher energy** in field

### General Quantum State

Any spin state can be written as a **superposition**:

$$|\psi\rangle = c_\alpha |\alpha\rangle + c_\beta |\beta\rangle = \begin{pmatrix} c_\alpha \\ c_\beta \end{pmatrix}$$

where $|c_\alpha|^2 + |c_\beta|^2 = 1$ (normalization).

### Physical Meaning

- $|c_\alpha|^2$ = probability of finding spin in $|\alpha\rangle$ state
- $|c_\beta|^2$ = probability of finding spin in $|\beta\rangle$ state
- The **phase** between $c_\alpha$ and $c_\beta$ matters (quantum coherence)!

### NMR Connection

At thermal equilibrium at temperature $T$:
$$\frac{n_\beta}{n_\alpha} = e^{-\Delta E/k_B T} \approx 1 - \frac{\gamma \hbar B_0}{k_B T}$$

The tiny population difference ($\sim 10^{-5}$ for ¹H at room temperature) is what we detect!

In [ ]:
# Define basis states
ket_alpha = np.array([[1], [0]], dtype=complex)  # Spin up |↑⟩
ket_beta = np.array([[0], [1]], dtype=complex)   # Spin down |↓⟩

# Bra vectors (for inner products)
bra_alpha = ket_alpha.conj().T
bra_beta = ket_beta.conj().T

print("Basis States:")
print(f"  |α⟩ = |↑⟩ = {ket_alpha.T[0]}")
print(f"  |β⟩ = |↓⟩ = {ket_beta.T[0]}")

# Verify orthonormality
print("\n✓ Orthonormality:")
print(f"  ⟨α|α⟩ = {(bra_alpha @ ket_alpha)[0,0]}")
print(f"  ⟨β|β⟩ = {(bra_beta @ ket_beta)[0,0]}")
print(f"  ⟨α|β⟩ = {(bra_alpha @ ket_beta)[0,0]}")

# Create example superposition states
# Equal superposition (maximum uncertainty in Iz)
psi_plus = (ket_alpha + ket_beta) / np.sqrt(2)  # Eigenstate of Ix with eigenvalue +1/2
psi_minus = (ket_alpha - ket_beta) / np.sqrt(2)  # Eigenstate of Ix with eigenvalue -1/2

# Eigenstate of Iy
psi_y_plus = (ket_alpha + 1j*ket_beta) / np.sqrt(2)

print("\n📊 Example Superposition States:")
print(f"  |+⟩ₓ = (|α⟩ + |β⟩)/√2 = {psi_plus.T[0]}")
print(f"  |−⟩ₓ = (|α⟩ − |β⟩)/√2 = {psi_minus.T[0]}")
print(f"  |+⟩ᵧ = (|α⟩ + i|β⟩)/√2 = {psi_y_plus.T[0]}")

In [ ]:
# Visualize spin states on the Bloch sphere
def bloch_sphere_plot(states_dict, title="Quantum Spin States on Bloch Sphere"):
    """
    Visualize spin-1/2 states on the Bloch sphere.
    states_dict: dictionary of {'label': state_vector}
    """
    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    # Draw the Bloch sphere
    u = np.linspace(0, 2 * np.pi, 100)
    v = np.linspace(0, np.pi, 100)
    x_sphere = np.outer(np.cos(u), np.sin(v))
    y_sphere = np.outer(np.sin(u), np.sin(v))
    z_sphere = np.outer(np.ones(np.size(u)), np.cos(v))
    ax.plot_surface(x_sphere, y_sphere, z_sphere, color='lightblue', alpha=0.15, linewidth=0)
    
    # Draw axes
    ax.plot([-1.2, 1.2], [0, 0], [0, 0], 'k-', alpha=0.3, linewidth=1)
    ax.plot([0, 0], [-1.2, 1.2], [0, 0], 'k-', alpha=0.3, linewidth=1)
    ax.plot([0, 0], [0, 0], [-1.2, 1.2], 'k-', alpha=0.3, linewidth=1)
    
    # Label axes
    ax.text(1.3, 0, 0, 'X', fontsize=12, fontweight='bold')
    ax.text(0, 1.3, 0, 'Y', fontsize=12, fontweight='bold')
    ax.text(0, 0, 1.3, 'Z (B₀)', fontsize=12, fontweight='bold')
    
    # Label poles
    ax.text(0, 0, 1.15, '|α⟩ (↑)', fontsize=11, ha='center', color='blue', fontweight='bold')
    ax.text(0, 0, -1.15, '|β⟩ (↓)', fontsize=11, ha='center', color='red', fontweight='bold')
    
    # Plot each state
    colors = plt.cm.rainbow(np.linspace(0, 1, len(states_dict)))
    for (label, state), color in zip(states_dict.items(), colors):
        # Extract coefficients
        c_alpha = state[0, 0]
        c_beta = state[1, 0]
        
        # Calculate Bloch vector components: r⃗ = ⟨ψ|σ⃗|ψ⟩
        r_x = 2 * np.real(c_alpha * np.conj(c_beta))
        r_y = 2 * np.imag(c_alpha * np.conj(c_beta))
        r_z = np.abs(c_alpha)**2 - np.abs(c_beta)**2
        
        # Draw state vector
        ax.quiver(0, 0, 0, r_x, r_y, r_z, color=color, arrow_length_ratio=0.15, linewidth=2.5)
        ax.text(r_x*1.1, r_y*1.1, r_z*1.1, label, fontsize=10, fontweight='bold', color=color)
    
    ax.set_xlim([-1.2, 1.2])
    ax.set_ylim([-1.2, 1.2])
    ax.set_zlim([-1.2, 1.2])
    ax.set_box_aspect([1,1,1])
    ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()

# Visualize our states
states_to_plot = {
    '|α⟩': ket_alpha,
    '|β⟩': ket_beta,
    '|+⟩ₓ': psi_plus,
    '|−⟩ₓ': psi_minus,
    '|+⟩ᵧ': psi_y_plus
}

bloch_sphere_plot(states_to_plot, "Spin-1/2 States Visualized on Bloch Sphere")

print("📊 Each arrow represents a pure quantum state.")
print("📊 The Z-axis is aligned with the magnetic field B₀.")
print("📊 NMR detects the X and Y components (transverse magnetization)!")

# Section 3: Tensor Products & Composite Quantum Systems

## 3.1 Building Two-Spin Systems

When we have **two coupled spins** (like in your J-coupled peaks!), we can't describe them independently. We need the **tensor product** to build the composite system.

### Mathematical Definition

For two spin-1/2 particles:
- Spin 1 lives in Hilbert space $\mathcal{H}_1$ (dimension 2)
- Spin 2 lives in Hilbert space $\mathcal{H}_2$ (dimension 2)
- Combined system lives in $\mathcal{H}_1 \otimes \mathcal{H}_2$ (dimension 4)

### Tensor Product of States

$$|i\rangle \otimes |j\rangle = |i, j\rangle = |ij\rangle$$

For two spins, we have **four basis states**:

$$\begin{align}
|\alpha\alpha\rangle &= |\uparrow\uparrow\rangle = \begin{pmatrix} 1 \\ 0 \end{pmatrix} \otimes \begin{pmatrix} 1 \\ 0 \end{pmatrix} = \begin{pmatrix} 1 \\ 0 \\ 0 \\ 0 \end{pmatrix} \\[0.5em]
|\alpha\beta\rangle &= |\uparrow\downarrow\rangle = \begin{pmatrix} 1 \\ 0 \end{pmatrix} \otimes \begin{pmatrix} 0 \\ 1 \end{pmatrix} = \begin{pmatrix} 0 \\ 1 \\ 0 \\ 0 \end{pmatrix} \\[0.5em]
|\beta\alpha\rangle &= |\downarrow\uparrow\rangle = \begin{pmatrix} 0 \\ 1 \end{pmatrix} \otimes \begin{pmatrix} 1 \\ 0 \end{pmatrix} = \begin{pmatrix} 0 \\ 0 \\ 1 \\ 0 \end{pmatrix} \\[0.5em]
|\beta\beta\rangle &= |\downarrow\downarrow\rangle = \begin{pmatrix} 0 \\ 1 \end{pmatrix} \otimes \begin{pmatrix} 0 \\ 1 \end{pmatrix} = \begin{pmatrix} 0 \\ 0 \\ 0 \\ 1 \end{pmatrix}
\end{align}$$

### NMR Connection

In your J-coupled multiplets, each sub-peak corresponds to a transition involving one of these four basis states! The splitting pattern directly reveals the quantum state structure.

In [ ]:
# Construct two-spin basis states using tensor products
from numpy import kron

# Two-spin product basis (4 states)
aa = kron(ket_alpha, ket_alpha)  # |↑↑⟩
ab = kron(ket_alpha, ket_beta)   # |↑↓⟩
ba = kron(ket_beta, ket_alpha)   # |↓↑⟩
bb = kron(ket_beta, ket_beta)    # |↓↓⟩

print("Two-Spin Product Basis States:\n")
print("|αα⟩ = |↑↑⟩:")
print(aa.T)
print("\n|αβ⟩ = |↑↓⟩:")
print(ab.T)
print("\n|βα⟩ = |↓↑⟩:")
print(ba.T)
print("\n|ββ⟩ = |↓↓⟩:")
print(bb.T)

# Verify orthonormality
print("\n✓ Orthonormality check:")
basis_2spin = [aa, ab, ba, bb]
labels = ['αα', 'αβ', 'βα', 'ββ']

overlap_matrix = np.zeros((4, 4))
for i in range(4):
    for j in range(4):
        overlap_matrix[i, j] = np.abs(basis_2spin[i].conj().T @ basis_2spin[j])[0, 0]

print(pd.DataFrame(overlap_matrix, columns=labels, index=labels))

## 3.2 Tensor Product of Operators

Operators on composite systems also use tensor products!

### Single-Spin Operators on Multi-Spin Systems

For spin 1 operator $\hat{A}_1$ acting only on spin 1:
$$\hat{A}_1 \otimes \hat{I}_2$$

For spin 2 operator $\hat{B}_2$ acting only on spin 2:
$$\hat{I}_1 \otimes \hat{B}_2$$

### Example: $I_z$ Total Spin Component

$$\hat{I}_z^{total} = \hat{I}_z^{(1)} \otimes \hat{I} + \hat{I} \otimes \hat{I}_z^{(2)}$$

### Composite Observables: The J-Coupling Hamiltonian

This is where NMR gets interesting! The J-coupling Hamiltonian couples the spins:

$$\hat{H}_J = 2\pi J \, (\hat{I}_z^{(1)} \otimes \hat{I}_z^{(2)} + \hat{I}_x^{(1)} \otimes \hat{I}_x^{(2)} + \hat{I}_y^{(1)} \otimes \hat{I}_y^{(2)})$$

This interaction **creates entanglement** between the spins! Let's compute it using your experimental J-coupling values.

In [ ]:
# Construct two-spin operators
I1z = kron(Iz, I)  # Iz for spin 1, identity for spin 2
I2z = kron(I, Iz)  # Identity for spin 1, Iz for spin 2

I1x = kron(Ix, I)
I2x = kron(I, Ix)

I1y = kron(Iy, I)
I2y = kron(I, Iy)

print("Two-Spin Operator Examples:\n")
print("I₁z ⊗ I₂ (Iz acting on spin 1):")
print(I1z)
print("\nI₁ ⊗ I₂z (Iz acting on spin 2):")
print(I2z)

# Total Iz operator
Iz_total = I1z + I2z
print("\nI_z^total = I₁z + I₂z:")
print(Iz_total)

# Verify eigenvalues (should be 1, 0, 0, -1 for the four basis states)
eigvals_Iz_total = np.linalg.eigvalsh(Iz_total)
print(f"\n✓ Eigenvalues of I_z^total: {eigvals_Iz_total}")
print("  These correspond to total angular momentum projections: m = +1, 0, 0, -1")

In [ ]:
# Build J-coupling Hamiltonian using experimental data
if quantum_data['j_couplings']:
    # Use the first detected J-coupling
    first_group = list(quantum_data['j_couplings'].keys())[0]
    J_Hz = quantum_data['j_couplings'][first_group]['J_avg_Hz']
    
    print(f"Using experimental J-coupling from: {first_group}")
    print(f"J = {J_Hz:.2f} Hz\n")
    
    # J-coupling Hamiltonian (scalar coupling)
    # H_J = 2πJ (I₁·I₂) = 2πJ (I₁xI₂x + I₁yI₂y + I₁zI₂z)
    H_J = 2 * np.pi * J_Hz * (I1x @ I2x + I1y @ I2y + I1z @ I2z)
    
    print("J-Coupling Hamiltonian H_J = 2πJ (I₁·I₂):")
    print(H_J.real)  # Should be real and symmetric
    
    # Diagonalize to find energy eigenstates
    eigvals_J, eigvecs_J = np.linalg.eigh(H_J)
    
    print(f"\n✓ Energy eigenvalues (in Hz):")
    for i, E in enumerate(eigvals_J / (2*np.pi)):
        print(f"  E_{i} = {E:.4f} Hz")
    
    print(f"\n📊 The J-coupling splits the energy levels!")
    print(f"📊 Energy difference between states: ΔE ≈ {(eigvals_J.max() - eigvals_J.min())/(2*np.pi):.2f} Hz")
    print(f"📊 This matches your observed peak splitting in the multiplet!")
    
else:
    # Use default J-coupling for demonstration
    J_Hz = 7.0
    H_J = 2 * np.pi * J_Hz * (I1x @ I2x + I1y @ I2y + I1z @ I2z)
    eigvals_J, eigvecs_J = np.linalg.eigh(H_J)
    print(f"Using default J = {J_Hz} Hz for demonstration\n")

In [ ]:
# Visualize energy level diagram
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: No coupling (product states)
ax1.hlines([1, 0, 0, -1], 0, 1, colors=['red', 'blue', 'blue', 'red'], linewidth=3)
ax1.text(1.1, 1, '|αα⟩', fontsize=12, va='center')
ax1.text(1.1, 0.05, '|αβ⟩', fontsize=12, va='center')
ax1.text(1.1, -0.05, '|βα⟩', fontsize=12, va='center')
ax1.text(1.1, -1, '|ββ⟩', fontsize=12, va='center')
ax1.set_ylabel('Energy / (ℏω₀/2)', fontsize=13, fontweight='bold')
ax1.set_title('Uncoupled Spins (J = 0)', fontsize=13, fontweight='bold')
ax1.set_xlim(-0.2, 1.8)
ax1.set_ylim(-1.5, 1.5)
ax1.set_xticks([])
ax1.grid(axis='y', alpha=0.3)

# Right: With J-coupling (eigenstates mixed)
E_scaled = eigvals_J / (np.pi * J_Hz)  # Normalize for comparison
colors_coupled = plt.cm.viridis(np.linspace(0, 1, 4))
for i, (E, color) in enumerate(zip(E_scaled, colors_coupled)):
    ax2.hlines(E, 0, 1, colors=color, linewidth=3)
    ax2.text(1.1, E, f'E_{i}', fontsize=12, va='center', color=color, fontweight='bold')

ax2.set_ylabel('Energy / (πJ)', fontsize=13, fontweight='bold')
ax2.set_title(f'J-Coupled Spins (J = {J_Hz:.1f} Hz)', fontsize=13, fontweight='bold')
ax2.set_xlim(-0.2, 1.8)
ax2.set_xticks([])
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("📊 Left: Without coupling, |αβ⟩ and |βα⟩ are degenerate (same energy)")
print("📊 Right: J-coupling breaks the degeneracy → observable splitting in NMR!")

## 3.3 Outer Products and Projection Operators

The **outer product** |ψ⟩⟨φ| creates an operator from two states.

### Definition

$$|\psi\rangle\langle\phi| = \begin{pmatrix} \psi_1 \\ \psi_2 \\ \vdots \end{pmatrix} \begin{pmatrix} \phi_1^* & \phi_2^* & \cdots \end{pmatrix}$$

This is an **operator** that acts on states, not just a number!

### Projection Operators

A special case is $\hat{P}_\psi = |\psi\rangle\langle\psi|$, which projects onto state $|\psi\rangle$:

$$\hat{P}_\psi |\phi\rangle = |\psi\rangle\langle\psi|\phi\rangle = \langle\psi|\phi\rangle |\psi\rangle$$

Properties:
- $\hat{P}_\psi^2 = \hat{P}_\psi$ (idempotent)
- $\text{Tr}(\hat{P}_\psi) = 1$
- $\langle\psi|\hat{P}_\psi|\psi\rangle = 1$ (probability of being in state $|\psi\rangle$)

### Completeness Relation

The basis states form a complete set:

$$\sum_i |i\rangle\langle i| = \hat{I}$$

For spin-1/2: $|\alpha\rangle\langle\alpha| + |\beta\rangle\langle\beta| = \hat{I}$

This is fundamental for density matrix formalism!

In [ ]:
# Compute outer products and projection operators

# Single-spin projectors
P_alpha = ket_alpha @ bra_alpha
P_beta = ket_beta @ bra_beta

print("Projection Operators:\n")
print("P_α = |α⟩⟨α|:")
print(P_alpha)
print("\nP_β = |β⟩⟨β|:")
print(P_beta)

# Verify completeness relation
completeness = P_alpha + P_beta
print("\n✓ Completeness: P_α + P_β = I?")
print(completeness)
print(f"  Equals identity? {np.allclose(completeness, I)}")

# Two-spin projector example
P_singlet = (ab - ba) / np.sqrt(2)  # Singlet state
projector_singlet = P_singlet @ P_singlet.conj().T

print("\n📊 Singlet State Projector:")
print("P_singlet = |Ψ⁻⟩⟨Ψ⁻| where |Ψ⁻⟩ = (|↑↓⟩ - |↓↑⟩)/√2")
print(projector_singlet.real)

# Verify idempotency
print(f"\n✓ Idempotency check: P² = P? {np.allclose(projector_singlet @ projector_singlet, projector_singlet)}")
print(f"✓ Trace = 1? {np.allclose(np.trace(projector_singlet), 1)}")

# Section 4: Density Matrices - Pure and Mixed Quantum States

## 4.1 What is a Density Matrix?

The **density operator** (or density matrix) $\hat{\rho}$ is the most general way to describe a quantum system, including:
- **Pure states**: Fully coherent quantum superpositions
- **Mixed states**: Statistical mixtures (classical uncertainty + quantum superposition)

### Density Matrix for Pure States

For a pure state $|\psi\rangle = c_\alpha|\alpha\rangle + c_\beta|\beta\rangle$:

$$\hat{\rho} = |\psi\rangle\langle\psi| = \begin{pmatrix} |c_\alpha|^2 & c_\alpha c_\beta^* \\ c_\beta c_\alpha^* & |c_\beta|^2 \end{pmatrix}$$

### Properties

1. **Hermitian**: $\hat{\rho} = \hat{\rho}^\dagger$
2. **Positive semi-definite**: All eigenvalues ≥ 0
3. **Normalized**: $\text{Tr}(\hat{\rho}) = 1$
4. **Pure state condition**: $\text{Tr}(\hat{\rho}^2) = 1$

### Physical Interpretation

- **Diagonal elements**: Populations (probabilities in basis states)
- **Off-diagonal elements**: Coherences (quantum phase relationships)

### NMR Connection

In NMR, we manipulate density matrices with RF pulses and measure them through signal detection! The FID directly encodes off-diagonal ("coherence") elements.

In [ ]:
# Compute density matrices for various states

def density_matrix(state_vector):
    """Compute density matrix from state vector"""
    return state_vector @ state_vector.conj().T

def purity(rho):
    """Compute purity Tr(ρ²) - equals 1 for pure states"""
    return np.real(np.trace(rho @ rho))

def visualize_density_matrix(rho, title, labels=None):
    """Visualize density matrix as heatmap (real and imaginary parts)"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    
    # Real part
    im1 = ax1.imshow(rho.real, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    ax1.set_title(f'{title} - Real Part', fontsize=13, fontweight='bold')
    if labels:
        ax1.set_xticks(range(len(labels)))
        ax1.set_yticks(range(len(labels)))
        ax1.set_xticklabels(labels)
        ax1.set_yticklabels(labels)
    plt.colorbar(im1, ax=ax1, fraction=0.046)
    
    # Imaginary part
    im2 = ax2.imshow(rho.imag, cmap='PRGn', vmin=-1, vmax=1, aspect='auto')
    ax2.set_title(f'{title} - Imaginary Part', fontsize=13, fontweight='bold')
    if labels:
        ax2.set_xticks(range(len(labels)))
        ax2.set_yticks(range(len(labels)))
        ax2.set_xticklabels(labels)
        ax2.set_yticklabels(labels)
    plt.colorbar(im2, ax=ax2, fraction=0.046)
    
    plt.tight_layout()
    plt.show()

# Pure states density matrices
rho_alpha = density_matrix(ket_alpha)
rho_beta = density_matrix(ket_beta)
rho_plus = density_matrix(psi_plus)
rho_yplus = density_matrix(psi_y_plus)

print("Single-Spin Density Matrices:\n")
print("ρ_α = |α⟩⟨α|:")
print(rho_alpha)
print(f"Purity: {purity(rho_alpha):.6f} (pure state: Tr(ρ²) = 1)\n")

print("ρ₊ₓ = |+⟩ₓ⟨+|ₓ (equal superposition):")
print(rho_plus)
print(f"Purity: {purity(rho_plus):.6f}\n")

# Visualize
visualize_density_matrix(rho_plus, '|+⟩ₓ State', labels=['α', 'β'])

print("📊 Off-diagonal elements = 0.5 indicate maximum coherence!")
print("📊 This represents a quantum superposition with definite phase.")

## 4.2 Mixed States and Thermal Ensembles

A **mixed state** is a statistical ensemble (classical probability distribution) over pure states:

$$\hat{\rho}_{\text{mixed}} = \sum_i p_i |\psi_i\rangle\langle\psi_i|$$

where $p_i$ are classical probabilities ($\sum_i p_i = 1$).

### Thermal Equilibrium in NMR

At temperature $T$, the ensemble follows Boltzmann distribution:

$$\hat{\rho}_{\text{thermal}} = \frac{e^{-\hat{H}/k_BT}}{Z} = \frac{e^{-\hbar\omega_0 \hat{I}_z/k_BT}}{\text{Tr}(e^{-\hbar\omega_0 \hat{I}_z/k_BT})}$$

For high temperature (always true in NMR!): $k_BT \gg \hbar\omega_0$, so:

$$\hat{\rho}_{\text{thermal}} \approx \frac{\hat{I}}{2} + \frac{\hbar\omega_0}{4k_BT}\hat{I}_z$$

This is almost the identity (maximum entropy) with a tiny bias toward spin-up!

### Purity Test

- Pure state: $\text{Tr}(\hat{\rho}^2) = 1$
- Mixed state: $\text{Tr}(\hat{\rho}^2) < 1$ (approaches $1/d$ for maximally mixed)

In [ ]:
# Create mixed states

# Completely mixed state (maximum entropy)
rho_mixed_max = I / 2  # Equal probability for α and β, no coherence
print("Maximally Mixed State (Identity/2):")
print(rho_mixed_max)
print(f"Purity: {purity(rho_mixed_max):.6f} (maximally mixed: Tr(ρ²) = 1/2)\n")

# Partially mixed: 70% |α⟩, 30% |β⟩
rho_partial = 0.7 * rho_alpha + 0.3 * rho_beta
print("Partially Mixed State (70% α, 30% β):")
print(rho_partial)
print(f"Purity: {purity(rho_partial):.6f} (between 0.5 and 1)\n")

# Thermal equilibrium at room temperature (very close to maximally mixed!)
# For ¹H at B₀ = 9.4 T, T = 300 K
B0 = quantum_data['B0_tesla']
k_B = 1.380649e-23  # J/K
h_bar = 1.054571817e-34  # J·s
gamma_H = 2.675e8  # rad/(s·T)
T = 300  # K

omega_0 = gamma_H * B0
polarization = h_bar * omega_0 / (4 * k_B * T)

rho_thermal = I/2 + polarization * Iz
print(f"Thermal Equilibrium at T = {T} K, B₀ = {B0:.2f} T:")
print(f"Polarization: {polarization:.2e}")
print(rho_thermal)
print(f"Purity: {purity(rho_thermal):.6f}")

print("\n📊 NMR operates on nearly maximally mixed states!")
print(f"📊 Only ~{polarization*1e6:.1f} ppm excess in |α⟩ state")
print("📊 But this tiny difference is what creates your detectable signal!")

In [ ]:
# Visualize pure vs mixed states
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

states_to_viz = [
    (rho_alpha, '|α⟩ (Pure)', ['α', 'β']),
    (rho_plus, '|+⟩ₓ (Pure Superposition)', ['α', 'β']),
    (rho_yplus, '|+⟩ᵧ (Pure with Phase)', ['α', 'β']),
    (rho_mixed_max, 'Maximally Mixed', ['α', 'β']),
    (rho_partial, 'Partial Mix (70/30)', ['α', 'β']),
    (rho_thermal, f'Thermal (T={T}K)', ['α', 'β'])
]

for idx, (rho, title, labels) in enumerate(states_to_viz):
    ax = axes[idx // 3, idx % 3]
    im = ax.imshow(rho.real, cmap='RdBu_r', vmin=-0.7, vmax=0.7, aspect='auto')
    ax.set_title(f'{title}\nPurity = {purity(rho):.4f}', fontsize=11, fontweight='bold')
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels)
    ax.set_yticklabels(labels)
    
    # Add values as text
    for i in range(len(labels)):
        for j in range(len(labels)):
            text = ax.text(j, i, f'{rho.real[i, j]:.3f}',
                          ha="center", va="center", color="black", fontsize=9)

plt.tight_layout()
plt.show()

print("📊 Pure states: Purity = 1, have off-diagonal coherences")
print("📊 Mixed states: Purity < 1, diagonal elements vary")
print("📊 Thermal state: Almost diagonal (coherences decay fast in reality)")

# Section 5: Quantum Entanglement

## 5.1 What is Entanglement?

**Entanglement** is a uniquely quantum phenomenon where two (or more) particles become correlated in such a way that:
1. They cannot be described independently
2. Measuring one instantly affects the other (non-local correlations)
3. The correlations are stronger than any classical theory allows

### Separable vs. Entangled States

A two-spin state is **separable** (not entangled) if it can be written as a product:
$$|\psi\rangle_{\text{sep}} = |\phi\rangle_1 \otimes |\chi\rangle_2$$

A state is **entangled** if it CANNOT be written this way.

### The Bell States (Maximally Entangled States)

The four Bell states form an orthonormal basis of maximally entangled two-spin states:

$$\begin{align}
|\Phi^+\rangle &= \frac{1}{\sqrt{2}}(|\alpha\alpha\rangle + |\beta\beta\rangle) \quad \text{(triplet)}\\
|\Phi^-\rangle &= \frac{1}{\sqrt{2}}(|\alpha\alpha\rangle - |\beta\beta\rangle) \quad \text{(triplet)}\\
|\Psi^+\rangle &= \frac{1}{\sqrt{2}}(|\alpha\beta\rangle + |\beta\alpha\rangle) \quad \text{(triplet)}\\
|\Psi^-\rangle &= \frac{1}{\sqrt{2}}(|\alpha\beta\rangle - |\beta\alpha\rangle) \quad \text{(singlet)}
\end{align}$$

### NMR Connection

J-coupling creates entanglement! The eigenstates of the J-coupling Hamiltonian are (approximately) Bell states. Your observed doublets and triplets encode this entanglement structure.

In [ ]:
# Construct Bell states
bell_phi_plus = (aa + bb) / np.sqrt(2)   # |Φ⁺⟩
bell_phi_minus = (aa - bb) / np.sqrt(2)  # |Φ⁻⟩
bell_psi_plus = (ab + ba) / np.sqrt(2)   # |Ψ⁺⟩ (triplet)
bell_psi_minus = (ab - ba) / np.sqrt(2)  # |Ψ⁻⟩ (singlet)

print("Bell States (Maximally Entangled Two-Spin States):\n")
print("|Φ⁺⟩ = (|↑↑⟩ + |↓↓⟩)/√2:")
print(bell_phi_plus.T)
print("\n|Φ⁻⟩ = (|↑↑⟩ − |↓↓⟩)/√2:")
print(bell_phi_minus.T)
print("\n|Ψ⁺⟩ = (|↑↓⟩ + |↓↑⟩)/√2 (triplet):")
print(bell_psi_plus.T)
print("\n|Ψ⁻⟩ = (|↑↓⟩ − |↓↑⟩)/√2 (singlet - total spin S=0):")
print(bell_psi_minus.T)

# Verify orthonormality
bell_states = [bell_phi_plus, bell_phi_minus, bell_psi_plus, bell_psi_minus]
bell_labels = ['Φ⁺', 'Φ⁻', 'Ψ⁺', 'Ψ⁻']

overlap_bell = np.zeros((4, 4))
for i in range(4):
    for j in range(4):
        overlap_bell[i, j] = np.abs(bell_states[i].conj().T @ bell_states[j])[0, 0]

print("\n✓ Bell State Orthonormality:")
print(pd.DataFrame(overlap_bell, columns=bell_labels, index=bell_labels).round(4))

In [ ]:
# Compare separable vs. entangled states

# Separable state (product state): both spins definitely up
rho_separable = density_matrix(aa)  # |↑↑⟩⟨↑↑|

# Entangled state: Bell singlet
rho_singlet = density_matrix(bell_psi_minus)

print("Density Matrix Comparison:\n")
print("Separable |↑↑⟩:")
print(rho_separable)
print(f"Purity: {purity(rho_separable):.6f}\n")

print("Entangled |Ψ⁻⟩ (singlet):")
print(rho_singlet.real)
print(f"Purity: {purity(rho_singlet):.6f}\n")

# Visualize both
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Separable
im1 = axes[0].imshow(rho_separable.real, cmap='RdBu_r', vmin=-1, vmax=1)
axes[0].set_title('Separable |↑↑⟩ (Product State)', fontsize=13, fontweight='bold')
axes[0].set_xticks(range(4))
axes[0].set_yticks(range(4))
axes[0].set_xticklabels(['αα', 'αβ', 'βα', 'ββ'])
axes[0].set_yticklabels(['αα', 'αβ', 'βα', 'ββ'])
plt.colorbar(im1, ax=axes[0], fraction=0.046)

# Entangled
im2 = axes[1].imshow(rho_singlet.real, cmap='RdBu_r', vmin=-1, vmax=1)
axes[1].set_title('Entangled |Ψ⁻⟩ (Bell Singlet)', fontsize=13, fontweight='bold')
axes[1].set_xticks(range(4))
axes[1].set_yticks(range(4))
axes[1].set_xticklabels(['αα', 'αβ', 'βα', 'ββ'])
axes[1].set_yticklabels(['αα', 'αβ', 'βα', 'ββ'])
plt.colorbar(im2, ax=axes[1], fraction=0.046)

plt.tight_layout()
plt.show()

print("📊 Separable: Only one basis state populated (single element = 1)")
print("📊 Entangled: Multiple basis states with quantum coherences (off-diagonal)")
print("📊 The pattern of the singlet density matrix is the signature of entanglement!")

## 5.2 Measuring Entanglement: Von Neumann Entropy

How do we quantify entanglement? One powerful measure is the **von Neumann entropy** of the **reduced density matrix**.

### Partial Trace

For a two-spin system with density matrix $\hat{\rho}_{12}$, we obtain the reduced density matrix for spin 1 by "tracing out" spin 2:

$$\hat{\rho}_1 = \text{Tr}_2(\hat{\rho}_{12}) = \sum_i \langle i|_2 \hat{\rho}_{12} |i\rangle_2$$

### Von Neumann Entropy

$$S(\hat{\rho}) = -\text{Tr}(\hat{\rho} \log_2 \hat{\rho}) = -\sum_i \lambda_i \log_2 \lambda_i$$

where $\lambda_i$ are the eigenvalues of $\hat{\rho}$.

### Entanglement Entropy

For a pure bipartite state $|\psi\rangle_{12}$:
- $S(\rho_1) = S(\rho_2)$ (entanglement entropy)
- $S = 0$: Separable (no entanglement)
- $S = 1$: Maximally entangled (Bell state for 2 qubits)

### NMR Application

The J-coupling strength determines how quickly entanglement builds up. Stronger J → faster entanglement generation!

In [ ]:
# Implement partial trace and entanglement entropy

def partial_trace_2(rho_12, keep=1):
    """
    Compute partial trace of 2-qubit density matrix.
    keep=1: trace out qubit 2, return rho_1
    keep=2: trace out qubit 1, return rho_2
    """
    if rho_12.shape != (4, 4):
        raise ValueError("Input must be 4x4 density matrix")
    
    rho_reduced = np.zeros((2, 2), dtype=complex)
    
    if keep == 1:
        # Trace out qubit 2: sum over |0⟩₂ and |1⟩₂
        rho_reduced[0, 0] = rho_12[0, 0] + rho_12[1, 1]  # ⟨0|₁ρ|0⟩₁
        rho_reduced[0, 1] = rho_12[0, 2] + rho_12[1, 3]  
        rho_reduced[1, 0] = rho_12[2, 0] + rho_12[3, 1]
        rho_reduced[1, 1] = rho_12[2, 2] + rho_12[3, 3]  # ⟨1|₁ρ|1⟩₁
    elif keep == 2:
        # Trace out qubit 1: sum over |0⟩₁ and |1⟩₁
        rho_reduced[0, 0] = rho_12[0, 0] + rho_12[2, 2]
        rho_reduced[0, 1] = rho_12[0, 1] + rho_12[2, 3]
        rho_reduced[1, 0] = rho_12[1, 0] + rho_12[3, 2]
        rho_reduced[1, 1] = rho_12[1, 1] + rho_12[3, 3]
    
    return rho_reduced

def von_neumann_entropy(rho, base=2):
    """Compute von Neumann entropy S(ρ) = -Tr(ρ log ρ)"""
    eigvals = np.linalg.eigvalsh(rho)
    eigvals = eigvals[eigvals > 1e-12]  # Filter out numerical zeros
    if base == 2:
        return -np.sum(eigvals * np.log2(eigvals))
    else:
        return -np.sum(eigvals * np.log(eigvals))

# Test on separable and entangled states
print("Entanglement Analysis:\n")

# Separable state |↑↑⟩
rho1_sep = partial_trace_2(rho_separable, keep=1)
S_sep = von_neumann_entropy(rho1_sep)
print("Separable state |↑↑⟩:")
print(f"  Reduced density matrix ρ₁:")
print(f"  {rho1_sep}")
print(f"  Entanglement entropy: S = {S_sep:.6f} bits")
print(f"  → No entanglement! (S = 0)\n")

# Entangled singlet state
rho1_singlet = partial_trace_2(rho_singlet, keep=1)
S_singlet = von_neumann_entropy(rho1_singlet)
print("Bell singlet |Ψ⁻⟩:")
print(f"  Reduced density matrix ρ₁:")
print(f"  {rho1_singlet}")
print(f"  Entanglement entropy: S = {S_singlet:.6f} bits")
print(f"  → Maximally entangled! (S = 1 bit = maximum for 2 qubits)\n")

print("✓ The reduced density matrix of an entangled state is MIXED!")
print("✓ Even though the full system is in a pure state, each subsystem looks mixed.")
print("✓ This is the signature of entanglement.")

## 5.3 Entanglement in NMR: J-Coupled Spins

Now let's connect this back to your NMR data! The J-coupling Hamiltonian creates entanglement between coupled spins.

### Time Evolution Under J-Coupling

Starting from a separable state $|\alpha\alpha\rangle$, the J-coupling evolves it:

$$|\psi(t)\rangle = e^{-i H_J t/\hbar} |\alpha\alpha\rangle$$

The entanglement grows with time until reaching a maximum, then oscillates.

### Entanglement Dynamics

$$S(t) = -\sum_i \lambda_i(t) \log_2 \lambda_i(t)$$

where $\lambda_i(t)$ are eigenvalues of the reduced density matrix at time $t$.

In [ ]:
# Simulate entanglement dynamics under J-coupling

# Time evolution under H_J
t_max = 1.0 / J_Hz  # One full J-coupling period
times = np.linspace(0, t_max, 200)

entropies = []
for t in times:
    # Time evolution operator
    U_t = linalg.expm(-1j * H_J * t)
    
    # Evolve initial state |↑↑⟩
    psi_t = U_t @ aa
    
    # Compute density matrix and reduced density matrix
    rho_t = density_matrix(psi_t)
    rho1_t = partial_trace_2(rho_t, keep=1)
    
    # Compute entanglement entropy
    S_t = von_neumann_entropy(rho1_t)
    entropies.append(S_t)

# Plot entanglement vs time
plt.figure(figsize=(12, 6))
plt.plot(times * J_Hz, entropies, linewidth=2.5, color='darkviolet')
plt.axhline(y=1.0, color='red', linestyle='--', linewidth=1.5, label='Maximum (Bell state)')
plt.xlabel('Time (units of 1/J)', fontsize=13, fontweight='bold')
plt.ylabel('Entanglement Entropy S (bits)', fontsize=13, fontweight='bold')
plt.title(f'Entanglement Growth from J-Coupling (J = {J_Hz:.1f} Hz)', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)
plt.legend(fontsize=11)
plt.ylim(-0.05, 1.1)
plt.tight_layout()
plt.show()

max_entropy = np.max(entropies)
time_to_max = times[np.argmax(entropies)]

print(f"📊 Maximum entanglement: S_max = {max_entropy:.4f} bits")
print(f"📊 Reached at time: t = {time_to_max*1e3:.2f} ms = {time_to_max*J_Hz:.3f} / J")
print(f"📊 The J-coupling CREATES entanglement between the spins!")
print(f"📊 This is why your multiplet peaks encode quantum correlations!")

# Section 6: Bell's Theorem & Non-Local Quantum Correlations

## 6.1 Locality vs. Quantum Mechanics

**Bell's Theorem** (1964) proves that NO local classical theory can reproduce all predictions of quantum mechanics.

### Local Realism

Classical (local realistic) theories assume:
1. **Realism**: Physical properties exist independent of measurement
2. **Locality**: No faster-than-light influences (spacelike separated measurements are independent)

### Bell's Inequality

For spacelike separated measurements on two particles with measurement angles $\theta_1, \theta_2$:

**Classical bound (CHSH inequality)**:
$$|S| \leq 2$$

where the CHSH parameter is:
$$S = |E(a,b) - E(a,b')| + |E(a',b) + E(a',b')|$$

### Quantum Violation!

Quantum mechanics predicts:
$$S_{\text{QM}} = 2\sqrt{2} \approx 2.828 > 2$$

This violation has been experimentally confirmed countless times. **Nature is non-local!**

### NMR Connection

While NMR doesn't achieve true spacelike separation, J-coupled spins can demonstrate Bell inequality violations in the correlations between measurement outcomes!

In [ ]:
# Compute Bell/CHSH inequality violation

def correlation_function(state, A, B):
    """
    Compute correlation E(A,B) = ⟨ψ|A⊗B|ψ⟩
    for operators A, B on a two-qubit state
    """
    AB = kron(A, B)
    return np.real((state.conj().T @ AB @ state)[0, 0])

def compute_CHSH_parameter(state, theta_a, theta_a_prime, theta_b, theta_b_prime):
    """
    Compute CHSH parameter S for given measurement angles.
    Measurements are along directions in XZ plane: cos(θ)σz + sin(θ)σx
    """
    # Define measurement operators
    def measurement_operator(theta):
        return np.cos(theta) * Iz + np.sin(theta) * Ix
    
    A = measurement_operator(theta_a)
    A_prime = measurement_operator(theta_a_prime)
    B = measurement_operator(theta_b)
    B_prime = measurement_operator(theta_b_prime)
    
    # Compute correlations
    E_ab = correlation_function(state, A, B)
    E_ab_prime = correlation_function(state, A, B_prime)
    E_a_prime_b = correlation_function(state, A_prime, B)
    E_a_prime_b_prime = correlation_function(state, A_prime, B_prime)
    
    # CHSH parameter
    S = np.abs(E_ab - E_ab_prime) + np.abs(E_a_prime_b + E_a_prime_b_prime)
    
    return S, (E_ab, E_ab_prime, E_a_prime_b, E_a_prime_b_prime)

# Optimal angles for maximum CHSH violation
theta_a = 0
theta_a_prime = np.pi / 2
theta_b = np.pi / 4
theta_b_prime = -np.pi / 4

print("Bell/CHSH Inequality Test:\n")
print(f"Measurement angles:")
print(f"  Alice: θ = {np.degrees(theta_a):.0f}°, θ' = {np.degrees(theta_a_prime):.0f}°")
print(f"  Bob:   θ = {np.degrees(theta_b):.0f}°, θ' = {np.degrees(theta_b_prime):.0f}°\n")

# Test on separable state |↑↑⟩
S_sep, corrs_sep = compute_CHSH_parameter(aa, theta_a, theta_a_prime, theta_b, theta_b_prime) 
print(f"Separable state |↑↑⟩:")
print(f"  CHSH parameter: S = {S_sep:.4f}")
print(f"  Classical bound: S ≤ 2")
print(f"  Violates Bell? {S_sep > 2}\n")

# Test on entangled singlet
S_singlet, corrs_singlet = compute_CHSH_parameter(bell_psi_minus, theta_a, theta_a_prime, theta_b, theta_b_prime)
print(f"Bell singlet |Ψ⁻⟩:")
print(f"  CHSH parameter: S = {S_singlet:.4f}")
print(f"  Quantum maximum: S = 2√2 ≈ 2.828")
print(f"  Violates Bell? {S_singlet > 2} ✓")
print(f"  → Non-local correlations! No local theory can explain this!\n")

print("🌟 This is one of the most profound results in physics!")
print("🌟 Your J-coupled spins exhibit these same non-local correlations!")

In [ ]:
# Visualize CHSH parameter vs angle for different states

angles_scan = np.linspace(0, np.pi, 50)
S_sing_scan = []
S_prod_scan = []

for angle_b in angles_scan:
    # Fixed angles for Alice, scan Bob's angle
    S_s, _ = compute_CHSH_parameter(bell_psi_minus, 0, np.pi/2, angle_b, angle_b - np.pi/2)
    S_p, _ = compute_CHSH_parameter(aa, 0, np.pi/2, angle_b, angle_b - np.pi/2)
    S_sing_scan.append(S_s)
    S_prod_scan.append(S_p)

# Plot
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(np.degrees(angles_scan), S_sing_scan, linewidth=3, label='Bell Singlet |Ψ⁻⟩', color='purple')
ax.plot(np.degrees(angles_scan), S_prod_scan, linewidth=3, label='Product State |↑↑⟩', color='gray')
ax.axhline(y=2, color='red', linestyle='--', linewidth=2, label='Classical Bound (S = 2)')
ax.axhline(y=2*np.sqrt(2), color='green', linestyle=':', linewidth=2, label='Quantum Maximum (S = 2√2)')

ax.fill_between(np.degrees(angles_scan), 2, 2*np.sqrt(2), alpha=0.2, color='red', label='Quantum Violation Region')

ax.set_xlabel('Bob\'s Measurement Angle θ_b (degrees)', fontsize=13, fontweight='bold')
ax.set_ylabel('CHSH Parameter S', fontsize=13, fontweight='bold')
ax.set_title('Bell Inequality Violation: Entangled vs. Separable States', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='best')
ax.grid(alpha=0.3)
ax.set_ylim(0, 3)
plt.tight_layout()
plt.show()

print("📊 For optimal angles, the singlet violates Bell's inequality!")
print("📊 The product state always satisfies the classical bound.")
print("📊 The gap between classical and quantum is the signature of entanglement.")

# Section 7: Heisenberg Uncertainty Principle

## 7.1 Uncertainty Relations for Non-Commuting Observables

The **Heisenberg Uncertainty Principle** states that certain pairs of physical quantities cannot be simultaneously known with arbitrary precision.

### General Form

For any two observables $\hat{A}$ and $\hat{B}$:

$$\Delta A \cdot \Delta B \geq \frac{1}{2}\left|\langle[\hat{A}, \hat{B}]\rangle\right|$$

where:
- $\Delta A = \sqrt{\langle \hat{A}^2 \rangle - \langle \hat{A} \rangle^2}$ is the standard deviation
- $[\hat{A}, \hat{B}] = \hat{A}\hat{B} - \hat{B}\hat{A}$ is the commutator

### Position-Momentum Uncertainty

The famous form: $\Delta x \cdot \Delta p \geq \frac{\hbar}{2}$

### Spin Component Uncertainty

For spin-1/2 particles:

$$\Delta I_x \cdot \Delta I_y \geq \frac{1}{2}\left|\langle I_z \rangle\right|$$

### Physical Meaning

- Non-commuting observables are "complementary" - precise knowledge of one implies complete uncertainty in the other
- This is NOT due to measurement disturbance - it's a fundamental property of quantum states!
- Minimum uncertainty states are called "coherent states"

### NMR Connection

The FID encodes uncertainties in spin components! The linewidth (frequency uncertainty) and decay time (time uncertainty) are related by Fourier uncertainty: $\Delta\omega \cdot \Delta t \sim 1$

In [ ]:
# Compute uncertainties for various spin states

def expectation_value(state, operator):
    """Compute ⟨ψ|A|ψ⟩"""
    return np.real((state.conj().T @ operator @ state)[0, 0])

def uncertainty(state, operator):
    """Compute ΔA = √(⟨A²⟩ - ⟨A⟩²)"""
    exp_A = expectation_value(state, operator)
    exp_A2 = expectation_value(state, operator @ operator)
    return np.sqrt(exp_A2 - exp_A**2)

# Test states
test_states = {
    '|α⟩ (spin up)': ket_alpha,
    '|β⟩ (spin down)': ket_beta,
    '|+⟩ₓ (x-eigenstate)': psi_plus,
    '|+⟩ᵧ (y-eigenstate)': psi_y_plus
}

print("Heisenberg Uncertainty Relations for Spin-1/2:\n")
print("="*80)

for name, state in test_states.items():
    # Compute expectation values
    Ix_avg = expectation_value(state, Ix)
    Iy_avg = expectation_value(state, Iy)
    Iz_avg = expectation_value(state, Iz)
    
    # Compute uncertainties
    Delta_Ix = uncertainty(state, Ix)
    Delta_Iy = uncertainty(state, Iy)
    Delta_Iz = uncertainty(state, Iz)
    
    # Compute commutator expectation values
    comm_xy = expectation_value(state, Ix @ Iy - Iy @ Ix)
    comm_yz = expectation_value(state, Iy @ Iz - Iz @ Iy)
    comm_zx = expectation_value(state, Iz @ Ix - Ix @ Iz)
    
    # Check uncertainty relations
    product_xy = Delta_Ix * Delta_Iy
    bound_xy = 0.5 * np.abs(comm_xy)
    
    product_yz = Delta_Iy * Delta_Iz
    bound_yz = 0.5 * np.abs(comm_yz)
    
    product_zx = Delta_Iz * Delta_Ix
    bound_zx = 0.5 * np.abs(comm_zx)
    
    print(f"\nState: {name}")
    print(f"  Expectation values: ⟨Ix⟩ = {Ix_avg:.4f}, ⟨Iy⟩ = {Iy_avg:.4f}, ⟨Iz⟩ = {Iz_avg:.4f}")
    print(f"  Uncertainties: ΔIx = {Delta_Ix:.4f}, ΔIy = {Delta_Iy:.4f}, ΔIz = {Delta_Iz:.4f}")
    print(f"\n  Uncertainty products:")
    print(f"    ΔIx·ΔIy = {product_xy:.4f} ≥ ½|⟨[Ix,Iy]⟩| = {bound_xy:.4f} ✓ {product_xy >= bound_xy - 1e-10}")
    print(f"    ΔIy·ΔIz = {product_yz:.4f} ≥ ½|⟨[Iy,Iz]⟩| = {bound_yz:.4f} ✓ {product_yz >= bound_yz - 1e-10}")
    print(f"    ΔIz·ΔIx = {product_zx:.4f} ≥ ½|⟨[Iz,Ix]⟩| = {bound_zx:.4f} ✓ {product_zx >= bound_zx - 1e-10}")

print("\n" + "="*80)
print("✓ All uncertainty relations satisfied!")
print("✓ Eigenstates have zero uncertainty in their measurement direction.")
print("✓ Maximum uncertainty (ΔI = 0.5) in perpendicular directions.")

In [ ]:
# Visualize uncertainty relations

# Create a general superposition: |ψ(θ,φ)⟩ = cos(θ/2)|α⟩ + e^(iφ)sin(θ/2)|β⟩
theta_range = np.linspace(0, np.pi, 50)
phi = 0  # Fix azimuthal angle

uncertainties_Ix = []
uncertainties_Iy = []
uncertainties_Iz = []
products_xy = []
bounds_xy = []

for theta in theta_range:
    psi_theta = np.cos(theta/2) * ket_alpha + np.exp(1j*phi) * np.sin(theta/2) * ket_beta
    
    Delta_Ix = uncertainty(psi_theta, Ix)
    Delta_Iy = uncertainty(psi_theta, Iy)
    Delta_Iz = uncertainty(psi_theta, Iz)
    
    uncertainties_Ix.append(Delta_Ix)
    uncertainties_Iy.append(Delta_Iy)
    uncertainties_Iz.append(Delta_Iz)
    
    product = Delta_Ix * Delta_Iy
    comm_exp = expectation_value(psi_theta, Ix @ Iy - Iy @ Ix)
    bound = 0.5 * np.abs(comm_exp)
    
    products_xy.append(product)
    bounds_xy.append(bound)

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Left: Individual uncertainties
ax1.plot(np.degrees(theta_range), uncertainties_Ix, linewidth=2.5, label='ΔIₓ', color='red')
ax1.plot(np.degrees(theta_range), uncertainties_Iy, linewidth=2.5, label='ΔIᵧ', color='blue')
ax1.plot(np.degrees(theta_range), uncertainties_Iz, linewidth=2.5, label='ΔIᵤ', color='green')
ax1.axhline(y=0.5, color='black', linestyle='--', linewidth=1, alpha=0.5, label='Maximum (0.5)')
ax1.set_xlabel('State Parameter θ (degrees)', fontsize=13, fontweight='bold')
ax1.set_ylabel('Uncertainty ΔI', fontsize=13, fontweight='bold')
ax1.set_title('Spin Component Uncertainties vs State', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(alpha=0.3)

# Right: Uncertainty product and bound
ax2.plot(np.degrees(theta_range), products_xy, linewidth=2.5, label='ΔIₓ · ΔIᵧ', color='purple')
ax2.plot(np.degrees(theta_range), bounds_xy, linewidth=2.5, linestyle='--', label='Bound: ½|⟨[Iₓ,Iᵧ]⟩|', color='red')
ax2.fill_between(np.degrees(theta_range), bounds_xy, 0, alpha=0.2, color='red', label='Forbidden Region')
ax2.set_xlabel('State Parameter θ (degrees)', fontsize=13, fontweight='bold')
ax2.set_ylabel('Uncertainty Product', fontsize=13, fontweight='bold')
ax2.set_title('Heisenberg Uncertainty Relation: ΔIₓ·ΔIᵧ ≥ ½|⟨[Iₓ,Iᵧ]⟩|', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("📊 Left: As you rotate the state on the Bloch sphere, uncertainties change")
print("📊 Right: The uncertainty product ALWAYS exceeds the quantum bound")
print("📊 Minimum uncertainty (touching the bound) for eigenstates θ=0° or 180°")

# Section 8: Synthesis - Quantum Mechanics in Your NMR Spectrum

## Bringing It All Together 🎯

Let's connect every quantum concept back to your actual NMR data!

### What Your Spectrum Tells Us About Quantum Mechanics

1. **Peak Positions (Chemical Shifts)** → Energy level separations between $|\alpha\rangle$ and $|\beta\rangle$
   - Each nucleus experiences different local magnetic field
   - Reflects electron shielding (quantum orbital structure)

2. **Peak Intensities** → Population differences in quantum states
   - Boltzmann distribution at thermal equilibrium
   - $n_\alpha - n_\beta \propto$ your observed signal

3. **Multiplet Splitting (J-coupling)** → Quantum entanglement!
   - J-coupling Hamiltonian: $H_J = 2\pi J \, \mathbf{I}_1 \cdot \mathbf{I}_2$
   - Creates superpositions of product states
   - Eigenstates are Bell states (or close to them)

4. **FID Decay (T₂ relaxation)** → Quantum decoherence
   - Loss of phase coherence (off-diagonal density matrix elements)
   - Environment-induced collapse of superposition

5. **Linewidths** → Heisenberg uncertainty: $\Delta \omega \cdot T_2 \sim 1$
   - Fourier relationship between time and frequency domains
   - Fundamental limit on precision

### The Quantum Picture

Your NMR measurement is detecting:
- **Coherences**: Off-diagonal elements of the density matrix (the FID signal itself!)
- **Transitions**: Changes between quantum eigenstates induced by RF pulses
- **Correlations**: Entanglement between J-coupled spins manifested as multiplets

In [ ]:
# Summary visualization: Connect NMR observables to quantum concepts

fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(3, 3, hspace=0.4, wspace=0.4)

# 1. Original NMR spectrum (top, spanning all columns)
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(ppm_axis, magnitude, linewidth=0.8, color='darkblue')
ax1.invert_xaxis()
ax1.set_xlim(12, 0)
ax1.set_xlabel('Chemical Shift δ (ppm)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Intensity', fontsize=12, fontweight='bold')
ax1.set_title('Your NMR Spectrum: A Window into Quantum Mechanics', fontsize=14, fontweight='bold')
ax1.grid(alpha=0.3)

# Annotate with quantum concepts
if quantum_data['j_couplings']:
    first_group = list(quantum_data['j_couplings'].keys())[0]
    center = quantum_data['j_couplings'][first_group]['center_ppm']
    ax1.annotate('J-coupling\n(Entanglement!)', xy=(center, magnitude.max()*0.8), 
                xytext=(center+1, magnitude.max()*0.9),
                arrowprops=dict(arrowstyle='->', color='red', lw=2),
                fontsize=11, fontweight='bold', color='red')

# 2. Density matrix (bottom left)
ax2 = fig.add_subplot(gs[1, 0])
im2 = ax2.imshow(rho_singlet.real, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax2.set_title('Entangled State\nDensity Matrix', fontsize=11, fontweight='bold')
ax2.set_xticks([])
ax2.set_yticks([])
plt.colorbar(im2, ax=ax2, fraction=0.046)

# 3. Entanglement evolution (bottom center)
ax3 = fig.add_subplot(gs[1, 1])
ax3.plot(times * J_Hz, entropies, linewidth=2, color='darkviolet')
ax3.set_xlabel('Time (1/J)', fontsize=10)
ax3.set_ylabel('Entanglement\nEntropy S', fontsize=10)
ax3.set_title('J-Coupling Creates\nEntanglement', fontsize=11, fontweight='bold')
ax3.grid(alpha=0.3)

# 4. Bell violation (bottom right)
ax4 = fig.add_subplot(gs[1, 2])
states_bell = ['Separable', 'Singlet']
S_values = [S_sep, S_singlet]
colors = ['gray', 'purple']
bars = ax4.bar(states_bell, S_values, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
ax4.axhline(y=2, color='red', linestyle='--', linewidth=2, label='Classical Limit')
ax4.axhline(y=2*np.sqrt(2), color='green', linestyle=':', linewidth=2, label='Quantum Max')
ax4.set_ylabel('CHSH Parameter S', fontsize=10)
ax4.set_title('Bell Inequality\nViolation', fontsize=11, fontweight='bold')
ax4.legend(fontsize=8)
ax4.set_ylim(0, 3)

# 5. Energy levels (bottom center)
ax5 = fig.add_subplot(gs[2, 0])
E_plot = eigvals_J / (2*np.pi)order = np.argsort(E_plot)
E_sorted = E_plot[order]
for i, E in enumerate(E_sorted):
    ax5.hlines(E, 0, 1, colors=plt.cm.viridis(i/4), linewidth=4)
ax5.set_ylabel('Energy (Hz)', fontsize=10)
ax5.set_title('Quantum Energy\nLevels', fontsize=11, fontweight='bold')
ax5.set_xticks([])
ax5.grid(axis='y', alpha=0.3)

# 6. Bloch sphere (bottom center)
ax6 = fig.add_subplot(gs[2, 1], projection='3d')
u = np.linspace(0, 2*np.pi, 20)
v = np.linspace(0, np.pi, 20)
x = np.outer(np.cos(u), np.sin(v))
y = np.outer(np.sin(u), np.sin(v))
z = np.outer(np.ones(np.size(u)), np.cos(v))
ax6.plot_surface(x, y, z, color='lightblue', alpha=0.1)
ax6.quiver(0, 0, 0, 0.5, 0.5, 0.7, color='red', arrow_length_ratio=0.2, linewidth=3)
ax6.set_xlim([-1, 1])
ax6.set_ylim([-1, 1])
ax6.set_zlim([-1, 1])
ax6.set_title('Quantum\nSuperposition', fontsize=11, fontweight='bold')
ax6.set_xticks([])
ax6.set_yticks([])
ax6.set_zticks([])

# 7. Uncertainty (bottom right)
ax7 = fig.add_subplot(gs[2, 2])
ax7.plot(np.degrees(theta_range), products_xy, linewidth=2, color='purple', label='ΔIₓ·ΔIᵧ')
ax7.plot(np.degrees(theta_range), bounds_xy, linewidth=2, linestyle='--', color='red', label='Bound')
ax7.fill_between(np.degrees(theta_range), bounds_xy, 0, alpha=0.2, color='red')
ax7.set_xlabel('State θ (deg)', fontsize=9)
ax7.set_ylabel('Uncertainty', fontsize=9)
ax7.set_title('Heisenberg\nUncertainty', fontsize=11, fontweight='bold')
ax7.legend(fontsize=8)
ax7.grid(alpha=0.3)

plt.suptitle('From NMR Data to Quantum Mechanics', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("🌟 COMPLETE QUANTUM PICTURE OF YOUR NMR SPECTRUM 🌟")
print("="*80)

## Summary of Quantum Concepts Explored

### ✅ What You've Learned

1. **Quantum States & Operators**
   - Spin-1/2 as simplest quantum system (2D Hilbert space)
   - Pauli matrices as spin operators
   - Basis states and superposition
   - Bloch sphere visualization

2. **Tensor Products & Composite Systems**
   - Building multi-spin states: $\mathcal{H}_1 \otimes \mathcal{H}_2$
   - Product basis: $|\alpha\alpha\rangle, |\alpha\beta\rangle, |\beta\alpha\rangle, |\beta\beta\rangle$
   - Composite operators and observables
   - J-coupling Hamiltonian creates correlations

3. **Density Matrices**
   - Pure states: $\hat{\rho} = |\psi\rangle\langle\psi|$, Tr($\rho^2$) = 1
   - Mixed states: statistical ensembles, Tr($\rho^2$) < 1
   - Thermal equilibrium in NMR (nearly maximally mixed!)
   - Diagonal = populations, off-diagonal = coherences

4. **Quantum Entanglement**
   - Separable vs. entangled states
   - Bell states (maximally entangled)
   - Von Neumann entropy as entanglement measure
   - Partial trace reveals mixed reduced states
   - J-coupling generates entanglement dynamically

5. **Bell's Theorem & Non-Locality**
   - CHSH inequality: classical bound S ≤ 2
   - Quantum violation: S = 2√2 for maximally entangled states
   - No local realistic theory can explain quantum correlations
   - Experimentally verified (Nobel Prize 2022!)

6. **Heisenberg Uncertainty Principle**
   - $\Delta A \cdot \Delta B \geq \frac{1}{2}|⟨[\hat{A},\hat{B}]⟩|$
   - Complementary observables cannot be simultaneously precise
   - Spin component uncertainties
   - Connected to FID linewidths via Fourier uncertainty

### 🎯 Key Insights

- **NMR is inherently quantum mechanical**: Every feature of your spectrum reflects quantum phenomena
- **J-coupling = entanglement**: The splitting patterns encode quantum correlations
- **Coherences are quantum**: The FID signal detects off-diagonal density matrix elements
- **Uncertainty is fundamental**: Not measurement error, but intrinsic quantum indeterminacy
- **Nature is non-local**: Bell violations show quantum correlations defy classical physics

### 🚀 Next Steps

Now you have the quantum mechanics foundation to understand:
- Advanced NMR techniques (2D NMR, COSY, NOESY)
- Quantum control and pulse sequences
- Quantum information processing using NMR
- Spin dynamics and relaxation theory
- Density operator formalism for NMR

**Keep exploring! The quantum world is beautiful and bizarre, and NMR gives you a direct window into it. 🌌**

In [ ]:
# Final summary: Print quantum data extracted from your NMR
print("\n" + "="*80)
print("📊 QUANTUM DATA EXTRACTED FROM YOUR NMR SPECTRUM")
print("="*80)
print(f"\n🔬 Magnetic Field: B₀ = {quantum_data['B0_tesla']:.2f} Tesla")
print(f"🔬 Spectrometer Frequency: ν₀ = {quantum_data['spectrometer_freq_MHz']:.2f} MHz")
print(f"🔬 Number of distinct spin systems: {len(quantum_data['peaks_ppm'])}")

if quantum_data['j_couplings']:
    print(f"\n🔗 J-Coupled Systems Detected: {len(quantum_data['j_couplings'])}")
    for group_name, data in quantum_data['j_couplings'].items():
        print(f"\n  {group_name}:")
        print(f"    Chemical shift: δ = {data['center_ppm']:.2f} ppm")
        print(f"    J-coupling constant: J = {data['J_avg_Hz']:.2f} Hz")
        print(f"    Entanglement strength: 2πJ = {2*np.pi*data['J_avg_Hz']:.2f} rad/s")
        print(f"    Number of sub-peaks: {data['n_peaks']}")
        print(f"    → Quantum interpretation: {data['n_peaks']}-level system")
        print(f"    → Creates entanglement on timescale: τ ~ 1/J = {1000/data['J_avg_Hz']:.1f} ms")

print("\n" + "="*80)
print("🌟 Your NMR spectrum is a quantum mechanical measurement!")
print("🌟 Every peak, every splitting, every linewidth tells a quantum story.")
print("="*80)

print("\n✨ Notebook complete! You've journeyed from FID to quantum entanglement. ✨")
print("📚 Review the visualizations above to solidify your understanding.")
print("🔬 Try modifying parameters to see how quantum predictions change!")
print("\n💡 Remember: NMR spectroscopy is one of the few techniques that lets you")
print("   directly observe and manipulate quantum coherences at room temperature!")
print("\n🎓 Happy learning! 🚀")